# Fitting g<sub>bs</sub> in C<sub>3</sub>-C<sub>4</sub> intermediates grown at 200 and 800 ppm CO<sub>2</sub>

## Setup

In [ ]:
import numpy as np
import pandas as pd
import math
from sympy import Symbol, Eq, solve, cse, lambdify
!pip install lmfit emcee corner
from lmfit import Parameters, Model
from google.colab import files, drive
from scipy import stats

from multiprocessing import Pool
import emcee
import corner  # For visualizing the posterior

import matplotlib.pyplot as plt
import os

# Mount Google Drive
drive.mount('/content/gdrive', force_remount=True)

# Define file names
WORKING_DIR_FILEPATH = 'gdrive/My Drive/Colab Notebooks/'
INPUT_ACI_FILENAME = 'merged_gas_ex_lmfit.csv'
INPUT_PARAMS_FILENAME = 'allocation_data_lmfit.csv'
INPUT_CSTARS_FILENAME = 'cstars.csv'
OUTPUT_CSV_FILENAME = 'fit_results_aggregated.csv'

# --- 1. Load the Plant Parameters data (f_GDC, f_Rd, f_rubisco) ---
df_organelle = pd.read_csv(WORKING_DIR_FILEPATH + INPUT_PARAMS_FILENAME)
# Assuming 'Species' and 'Plant' uniquely identify a set of organelle parameters
df_organelle.set_index(['Species', 'Plant'], inplace=True)
df_organelle.sort_index(inplace=True) # Sort index to fix PerformanceWarning
print(f"\nSuccessfully loaded Organelle Parameters (f_...) from {INPUT_PARAMS_FILENAME}. Head of data:")
print(df_organelle.head())

# --- 2. Load the R_d data from cstars.csv ---
df_cstars = pd.read_csv(WORKING_DIR_FILEPATH + INPUT_CSTARS_FILENAME)

# Fix potential column name mismatch for R_d (e.g., 'Rd' vs 'R_d')
df_cstars.rename(columns={'Rd': 'R_d'}, inplace=True)
df_cstars.set_index(['Species', 'Plant'], inplace=True)
df_cstars.sort_index(inplace=True) # Sort index to fix PerformanceWarning

print(f"\nSuccessfully loaded R_d values from {INPUT_CSTARS_FILENAME}. Head of data:")
print(df_cstars.head())
# Verify R_d is actually present
if 'R_d' not in df_cstars.columns:
    print("!!! WARNING: Column 'R_d' NOT FOUND in cstars.csv. Available columns:", df_cstars.columns.tolist())

# --- 3. Load the A-Ci curve data ---
df_aci = pd.read_csv(WORKING_DIR_FILEPATH + INPUT_ACI_FILENAME)

# Rename 'GrowthCO2ppm' to 'CO2' for internal script consistency
if 'GrowthCO2ppm' in df_aci.columns:
    df_aci.rename(columns={'GrowthCO2ppm': 'CO2'}, inplace=True)
else:
    print("\nWARNING: 'GrowthCO2ppm' column not found in ACI data. Assuming 'CO2' column exists for grouping.")

print(f"\nSuccessfully loaded A-Ci data from {INPUT_ACI_FILENAME} and renamed columns.")
print("A-Ci Data Columns:", df_aci.columns.tolist())
print(df_aci.head())


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.7/97.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 4.8 MB/s eta 0:00:00
Mounted at /content/gdrive

Successfully loaded Organelle Parameters (f_...) from allocation_data_lmfit.csv. Head of data:
                       CO2      f_GDC       f_Rd  f_rubisco
Species         Plant                                      
F. angustifolia 1      200  81.631409  42.483638  11.134180
                1      800  79.495928  35.116643  10.682031
                2      200  84.619985  39.830476  13.199709
                2      800  77.387221  45.478388  11.057149
                3      200  90.634349  50.073084  13.227624

Successfully loaded R_d values from cstars.csv. Head of data:
                       GrowthCO2ppm       R_d      Slope      Cstar
Species         Plant                                              
F. angustifolia 1       

## Define function that fits the von Caemmerer model for C2 photosynthesis

In [ ]:


def fit_c2_photosynthesis_model_bayesian(
        A_data,
        Ci_data,
        Ca_data,
        plant_params_data,
        fit_params_initial,
        universal_constants_data,
        mcmc_steps=2000,
        burn_in=200
):
    """
    Fits the von Caemmerer model for C2 photosynthesis using Bayesian MCMC (emcee).

    Args:
        A_data (list/numpy.array): Measured net photosynthesis rate (A).
        Ci_data (list/numpy.array): Intercellular CO2 concentration (Ci).
        Ca_data (list/numpy.array): Ambient CO2 concentration (Ca).
        plant_params_data (dict): Dictionary of known/assigned plant-specific parameters (species mean).
        fit_params_initial (dict): Dictionary of parameters to be fitted (V_cmax, g_bs).
        universal_constants_data (dict): Universal constants (e.g., Kc, gm).
        mcmc_steps (int): Total number of steps for MCMC simulation.
        burn_in (int): Number of steps to discard as burn-in.

    Returns:
        dict: A dictionary containing the fitted results, modelled curve, and MCMC samples.
    """

    print("--- Starting C2 Photosynthesis Model Fitting (Bayesian MCMC) ---")

    # =========================================================================
    # 1-4. SYMPY SETUP AND EQUATIONS (UNCHANGED)
    # =========================================================================

    # 1. Define SymPy Symbols
    # Photosynthesis rates and intermediate variables
    A, Vc, Vo, Rd = Symbol("A", positive=True), Symbol("V_c", positive=True), Symbol("V_o", positive=True), Symbol("R_d", positive=True)
    Am, Abs = Symbol("A_m", positive=True), Symbol("A_bs", positive=True)
    Vcm, Vom, Rdm = Symbol("V_cm", positive=True), Symbol("V_om", positive=True), Symbol("R_dm", positive=True)
    Vcbs, Vobs, Rdbs = Symbol("V_cbs", positive=True), Symbol("V_obs", positive=True), Symbol("R_dbs", positive=True)
    L, fGDC = Symbol("L", positive=True), Symbol("f_GDC", positive=True)

    # Conductances and Concentrations
    gbs, Cm, Cbs = Symbol("g_bs", positive=True), Symbol("C_m", positive=True), Symbol("C_bs", positive=True)
    Ci, Ca, gm, gs = Symbol("C_i", positive=True), Symbol("C_a", positive=True), Symbol("g_m", positive=True), Symbol("g_s", positive=True)
    Om, Obs = Symbol("O_m", positive=True), Symbol("O_bs", positive=True)

    # Enzyme and Respiration Properties
    Sco, Kc, Ko = Symbol("S_c/o", positive=True), Symbol("K_c", positive=True), Symbol("K_o", positive=True)
    Vcmax, Vcmaxm, Vcmaxbs = Symbol("V_cmax", positive=True), Symbol("V_cmaxm", positive=True), Symbol("V_cmaxbs", positive=True)
    frubisco, alpha = Symbol("f_rubisco", positive=True), Symbol("alpha", positive=True)
    fRd, Rdh, fRdh = Symbol("f_Rd", positive=True), Symbol("R_dh", positive=True), Symbol("f_Rdh", positive=True)

    # 2. Define the Master Symbol Map
    symbol_map = {
        "S_c/o": Sco, "K_c": Kc, "K_o": Ko, "g_m": gm,
        "alpha": alpha, "f_Rdh": fRdh, "O_m": Om,
        "f_GDC": fGDC, "f_Rd": fRd, "f_rubisco": frubisco, "R_d": Rd
    }

    # 3. Handle and Merge Constants
    merged_constants_str = universal_constants_data.copy()
    merged_constants_str.update(plant_params_data)

    full_constants_sympy = {}
    for str_key, val in merged_constants_str.items():
        if str_key in symbol_map:
            full_constants_sympy[symbol_map[str_key]] = val
        elif str_key not in ['V_cmax', 'g_bs']:
            raise ValueError(f"Unknown constant or parameter provided: {str_key}")

    gm_value = full_constants_sympy[gm]

    # 4. Define Equations (C-limited case)
    A_eq1 = Eq(A, Vc - 0.5 * Vo - Rd)
    Abs_eq2 = Eq(Abs, 0.5 * fGDC * Vom - L)
    L_eq1 = Eq(L, Rdbs + 0.5 * Vobs - Vcbs + 0.5 * fGDC * Vom)
    L_eq2 = Eq(L, gbs * (Cbs - Cm))
    L_eq3 = L_eq1.subs({L: L_eq2.rhs})
    Rdh_eq1 = Eq(Rdh, fRdh * Rd)
    Rdm_eq1 = Eq(Rdm, (1 - fRd) * (Rd - Rdh))
    Rdbs_eq1 = Eq(Rdbs, fRd * (Rd - Rdh))
    Cm_eq1 = Eq(Cm, Ci - A / gm)
    Obs_eq1 = Eq(Obs, Om + alpha * Abs / (0.047 * gbs))
    Vom_eq1 = Eq(Vom, 2 * Vcm * (0.5 / Sco) * (Om / Cm))
    Vobs_eq1 = Eq(Vobs, 2 * Vcbs * (0.5 / Sco) * (Obs / Cbs))
    Vcm_eq1 = Eq(Vcm, Vcmaxm * Cm / (Cm + Kc * (1 + Om / Ko)))
    Vcbs_eq1 = Eq(Vcbs, Vcmaxbs * Cbs / (Cbs + Kc * (1 + Obs / Ko)))
    Vcmaxm_eq1 = Eq(Vcmaxm, (1 - frubisco) * Vcmax)
    Vcmaxbs_eq1 = Eq(Vcmaxbs, frubisco * Vcmax)


    # 5. Formulate the General Expression for Net Photosynthesis (A)
    # ... (Cbs, Obs, Vcm_expr, Vcbs_expr, Vom_expr derivations) ...

    # Cbs
    Cbs_eq1 = L_eq3.subs(
        {Rdbs: Rdbs_eq1.rhs, Vobs: Vobs_eq1.rhs}
        ).subs(
            {Rdh: Rdh_eq1.rhs, Vcbs: Vcbs_eq1.rhs}
        ).subs(
            {Obs: Obs_eq1.rhs}
        ).subs(
            {Abs: Abs_eq2.rhs}
        ).subs(
            {Vom: Vom_eq1.rhs, L: L_eq2.rhs}
        ).subs(
            {Vcm: Vcm_eq1.rhs}
        ).subs(
            {Vcmaxm: Vcmaxm_eq1.rhs, Vcmaxbs: Vcmaxbs_eq1.rhs}
        ).subs(
            full_constants_sympy
        )
    Cbs_sol = solve(Cbs_eq1, Cbs)
    test_sols = [sol.subs({Vcmax: 100, Cm: 300, gbs: 0.0055}) for sol in Cbs_sol]
    if len(Cbs_sol) > 1:
        Cbs_sol_index = [index for index, sol in enumerate(test_sols) if sol > 0][0]

    # Obs
    Obs_eq1 = Obs_eq1.subs(
            {Abs: Abs_eq2.rhs}
        ).subs(
            {L: L_eq2.rhs, Vom: Vom_eq1.rhs}
        ).subs(
            {Cbs: Cbs_sol[Cbs_sol_index], Vcm: Vcm_eq1.rhs}
        ).subs(
            {Vcmaxm: Vcmaxm_eq1.rhs}
        ).subs(
            full_constants_sympy
        )
    # Wcm
    Vcm_eq2 = Vcm_eq1.subs(
        {Vcmaxm: Vcmaxm_eq1.rhs}
        ).subs(
            full_constants_sympy
        )
    Vcm_expr = Vcm_eq2.rhs

    # Wccbs
    Vcbs_eq2 = Vcbs_eq1.subs(
        {Vcmaxbs: Vcmaxbs_eq1.rhs, Obs: Obs_eq1.rhs}
        ).subs(
            {Cbs: Cbs_sol[Cbs_sol_index]}
        ).subs(
            full_constants_sympy
        )
    Vcbs_expr = Vcbs_eq2.rhs

    # Vom
    Vom_expr = Vom_eq1.rhs.subs(
        {Vcm: Vcm_expr}
        ).subs(
            full_constants_sympy
        )

    # Cbs - final
    Cbs_eq3 = L_eq3.subs(
        {Rdbs: Rdbs_eq1.rhs, Vobs: Vobs_eq1.rhs}
        ).subs(
            {Rdh: Rdh_eq1.rhs, Obs: Obs_eq1.rhs}
        ).subs(
            {Abs: Abs_eq2.rhs}
        ).subs(
            {Vom: Vom_eq1.rhs, L: L_eq2.rhs}
        ).subs(
            {Vcm: Vcm_expr, Vcbs: Vcbs_expr}
        ).subs(
            full_constants_sympy
        )

    ## Solve Cbs (using cse for stability)
    Cbs_replacements, Cbs_reduced1 = cse(Cbs_eq3)
    Cbs_sol_reduced = solve(Cbs_reduced1, Cbs)
    Cbs_sol_final = []
    for i in range(len(Cbs_sol_reduced)):
        Cbs_sol_final.append(Cbs_sol_reduced[i][0])
    Cbs_sol_final_subbed = []
    for sol in Cbs_sol_final:
        sol_subbed = sol
        for x, expr in reversed(Cbs_replacements):
            sol_subbed = sol_subbed.subs(x, expr)
        Cbs_sol_final_subbed.append(sol_subbed)
    test_sols_final = []
    for sol in Cbs_sol_final_subbed:
        sol_eval = sol.subs({
            Cm: 300, Vcmax: 100, Om: 210000, gbs: 0.0055,
        })
        test_sols_final.append(sol_eval)

    # Pick the positive one
    Cbs_expr = None
    for expr in test_sols_final:
        if expr > 0:
            Cbs_expr = expr
            break

    # Vobs
    Vobs_expr = Vobs_eq1.rhs.subs(
            {Vcbs: Vcbs_expr, Obs: Obs_eq1.rhs}
        ).subs(
            {Abs: Abs_eq2.rhs}
        ).subs(
            {Vom: Vom_eq1.rhs, L: L_eq2.rhs}
        ).subs(
            {Vcm: Vcm_expr, Cbs: Cbs_expr}
        )

    # A (Final Net Photosynthesis Expression)
    A_expr = Vc - 0.5*(Vom_eq1.rhs + Vobs_expr) - Rdm_eq1.rhs - Rdbs_eq1.rhs
    A_expr = A_expr.subs(
        {Vc: Vcbs_expr + Vcm_expr, Rdh: Rdh_eq1.rhs, Vcbs: Vcbs_expr, Vom: Vom_eq1.rhs}
        ).subs(
            {Vcm: Vcm_expr}
        ).subs(full_constants_sympy)

    # Abs: Bundle Sheath Net CO2 Fixation Rate (V_cbs - 0.5*V_obs - R_dbs)
    Abs_expr = Vcbs_expr - 0.5*Vobs_expr - Rdbs_eq1.rhs
    Abs_expr = Abs_expr.subs(
        {Rdh: Rdh_eq1.rhs,
        Vcm: Vcm_expr,
        Vom: Vom_eq1.rhs}
        ).subs(full_constants_sympy)


    # 5. Convert to Function
    ordered_fit_symbols = [Vcmax, gbs]
    print(f"--- 5. Lambdifying expression for params: {[s.name for s in ordered_fit_symbols]} ---")
    A_func = lambdify([Cm] + ordered_fit_symbols, A_expr, modules='numpy')

    # NOTE: We do NOT lambdify Abs_expr here because it's only needed for a single point calculation later.


    # 6. Data Prep (UNCHANGED)
    Cms_data_all = np.array(Ci_data) - np.array(A_data) / gm_value
    sorted_pairs = sorted(zip(A_data, Cms_data_all), key=lambda x: x[0])
    As_data, Cms_data = zip(*sorted_pairs)
    As_data, Cms_data = np.array(As_data), np.array(Cms_data)


    # Define walkers
    ndim = 3  # V_cmax, g_bs, sigma
    nwalkers = 100

    # 7-8. BAYESIAN SETUP AND RUN (UNCHANGED)
    # ... (log_prior, log_likelihood, log_probability, and MCMC run code remains the same) ...
    print(f"--- 8. Running MCMC (Walkers: {nwalkers}, Steps: {mcmc_steps}) ---")


    # Define log prior
    def log_prior(theta):
        """Uniform priors within specified bounds"""
        vcmax, gbs, sigma = theta

        # Get bounds from fit_params_initial
        vcmax_min = fit_params_initial['V_cmax']['min']
        vcmax_max = fit_params_initial['V_cmax']['max']
        gbs_min = fit_params_initial['g_bs']['min']
        gbs_max = fit_params_initial['g_bs']['max']

        if (vcmax_min < vcmax < vcmax_max and
            gbs_min < gbs < gbs_max and
            sigma > 0):
            return 0.0
        return -np.inf

    # Define log likelihood
    def log_likelihood(theta, Cm_obs, A_obs):
        """Calculate log likelihood for observed data"""
        vcmax, gbs, sigma = theta

        # Predict A using the model
        try:
            A_pred = A_func(Cm_obs, vcmax, gbs)

            # Handle potential numerical issues
            if not np.all(np.isfinite(A_pred)):
                return -np.inf

            # Gaussian likelihood
            residuals = A_obs - A_pred
            log_like = -0.5 * np.sum((residuals / sigma)**2 + np.log(2 * np.pi * sigma**2))
            return log_like
        except:
            return -np.inf

    # Define log probability (posterior)
    def log_probability(theta, Cm_obs, A_obs):
        """Calculate log posterior probability"""
        lp = log_prior(theta)
        if not np.isfinite(lp):
            return -np.inf
        return lp + log_likelihood(theta, Cm_obs, A_obs)

    # Initial guesses from fit_params_initial
    vcmax_init = fit_params_initial['V_cmax']['value']
    gbs_init = fit_params_initial['g_bs']['value']
    sigma_init = np.std(As_data) * 0.1  # Initial guess for noise

    # Add small random perturbations to initial position
    pos = np.array([vcmax_init, gbs_init, sigma_init]) + 1e-4 * np.random.randn(nwalkers, ndim)

    # Ensure all walkers start in valid prior region
    for i in range(nwalkers):
        while not np.isfinite(log_prior(pos[i])):
            pos[i] = np.array([vcmax_init, gbs_init, sigma_init]) + 1e-4 * np.random.randn(ndim)

    # Run MCMC
    sampler = emcee.EnsembleSampler(
        nwalkers,
        ndim,
        log_probability,
        args=(Cms_data, As_data)
    )

    # Run the MCMC sampler
    sampler.run_mcmc(pos, mcmc_steps, progress=True)

    # =========================================================================
    # 9. PROCESS RESULTS (CALCULATING GLY AND ABS)
    # =========================================================================
    print("--- 9. Processing MCMC Chains ---")

    # Discard burn-in and flatten chains
    flat_samples = sampler.get_chain(discard=burn_in, flat=True)

    # Calculate statistics (Median and 1-sigma quantiles)
    results_stats = {}
    v_mcmc = np.percentile(flat_samples[:, 0], [16, 50, 84])
    results_stats['V_cmax'] = {'value': v_mcmc[1], 'stderr': (v_mcmc[2]-v_mcmc[0])/2, 'lower_CI': v_mcmc[0], 'upper_CI': v_mcmc[2]}
    g_mcmc = np.percentile(flat_samples[:, 1], [16, 50, 84])
    results_stats['g_bs'] = {'value': g_mcmc[1], 'stderr': (g_mcmc[2]-g_mcmc[0])/2, 'lower_CI': g_mcmc[0], 'upper_CI': g_mcmc[2]}
    s_mcmc = np.percentile(flat_samples[:, 2], [16, 50, 84])

    # === Glycine Shuttle Rate and Abs Calculation ===

    ambient_index = None

    # 1. Search for a Ca step >= 380 ppm (robust ambient check)
    for idx, ca in enumerate(Ca_data):
        ca_rounded = round(ca, -1)
        if ca_rounded >= 380:
            ambient_index = idx
            break

    # Critical Check: Handle case where ambient_index is still None (e.g., if only low CO2 data is present)
    if ambient_index is None:
        ambient_index = np.argmin(np.abs(np.array(Ca_data) - 400))
        print(f"WARNING: Ideal ambient Ca not strictly found. Using closest point to 400 ppm (Index {ambient_index}).")

    A_ref = A_data[ambient_index]
    Ci_ref = Ci_data[ambient_index]
    Ca_ref = Ca_data[ambient_index]

    gm_val = universal_constants_data['g_m']
    Cm_gly_rate = Ci_ref - (A_ref / gm_val)

    print(f"Using Cm = {Cm_gly_rate:.2f} (from Ca={Ca_ref:.0f} ppm step) for rate calculations.")

    # 1. Substitute median fitted parameters and the specific Cm into the Vom expression
    Vom_calculated_expr = Vom_expr.subs(
        {Vcmax: results_stats['V_cmax']['value'],
         gbs: results_stats['g_bs']['value'],
         Cm: Cm_gly_rate}
        )

    # 2. Glycine Shuttle Rate = 0.5 * f_GDC * Vom
    gly_rate = Vom_calculated_expr * 0.5 * full_constants_sympy[fGDC]

    # 3. Calculate Abs (Bundle Sheath Net Fixation Rate) at this reference Cm
    Abs_ambient = Abs_expr.subs(
        {Vcmax: results_stats['V_cmax']['value'],
         gbs: results_stats['g_bs']['value'],
         Cm: Cm_gly_rate}
        )

    # 4. Store the results
    results_stats['Glycine_Shuttle_Rate'] = {'value': float(gly_rate)}
    results_stats['Cm_Used_for_GlyRate'] = Cm_gly_rate
    results_stats['Abs_at_Ambient_Cm'] = {'value': float(Abs_ambient)} # ADDED ABS HERE

    # === End Glycine Shuttle Rate and Abs Calculation ===

    # Generate Model Curve using Median Parameters
    print("--- 10. Generating Modelled Curve ---")
    # Cms_model_range = np.linspace(min(Cms_data), max(Cms_data) * 1.5, 500)
    Cms_model_range = np.arange(0, 351, 1)

    A_model = A_func(Cms_model_range, results_stats['V_cmax']['value'], results_stats['g_bs']['value'])

    # Add extra stats to match previous output format
    results_stats['stats'] = {
        'chisqr': "N/A (Bayesian)",
        'n_data': len(As_data),
        'sigma_estimated': s_mcmc[1] # Store median noise estimate
    }

    return {
        'A_model': A_model.tolist(),
        'Cms_data': Cms_data.tolist(),
        'Cm_model_range': Cms_model_range.tolist(),
        'fit_results': results_stats,
        'mcmc_samples': flat_samples
    }

## Loop through input data to fit model for each plant


### Parallel

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
import copy

# =============================================================================
# 1. SETUP CONSTANTS & DIRECTORIES
# =============================================================================

OVERWRITE = False

if 'WORKING_DIR_FILEPATH' not in locals():
    WORKING_DIR_FILEPATH = '.'

MCMC_STEPS = 1000
BURN_IN = 500

# Base defaults
DEFAULT_FIT_PARAMS_INITIAL = {
    'V_cmax': {'value': 50, 'min': 25, 'max': 250},
    'g_bs': {'value': 0.001, 'min': 1e-9, 'max': 0.2}
}

DEFAULT_UNIVERSAL_CONSTANTS = {
    'S_c/o': 76 * (0.0334 / 0.00126),
    'K_c': 14 / 0.0334,
    'K_o': 600 / 0.00126,
    'g_m': 0.6,
    'alpha': 1,
    'f_Rdh': 0.1,
    'O_m': 210000
}

# Directories
temp_param_dir = os.path.join(WORKING_DIR_FILEPATH, "temp_results_params")
temp_curve_dir = os.path.join(WORKING_DIR_FILEPATH, "temp_results_curves")
fig_dir = os.path.join(WORKING_DIR_FILEPATH, "Fit_Curves_Individual")

os.makedirs(temp_param_dir, exist_ok=True)
os.makedirs(temp_curve_dir, exist_ok=True)
os.makedirs(fig_dir, exist_ok=True)


# =============================================================================
# 2. PRE-CALCULATE SPECIES MEANS
# =============================================================================
print("📊 Calculating Species Means for Inputs...")

# 1. Organelle Means
# Reset index to ensure Species/Plant are columns, then group by Species & CO2
if 'Species' not in df_organelle.columns:
    df_org_temp = df_organelle.reset_index()
else:
    df_org_temp = df_organelle.copy()

# Group by Species and CO2 to get the mean across all plants
df_organelle_mean = df_org_temp.groupby(['Species', 'CO2']).mean(numeric_only=True).reset_index()
# Set Index for fast lookup: .loc[(Species, CO2)]
df_organelle_mean = df_organelle_mean.set_index(['Species', 'CO2'])


# 2. C-Stars (Rd) Means
if 'Species' not in df_cstars.columns:
    df_cs_temp = df_cstars.reset_index()
else:
    df_cs_temp = df_cstars.copy()

# Group by Species and GrowthCO2ppm
df_cstars_mean = df_cs_temp.groupby(['Species', 'GrowthCO2ppm']).mean(numeric_only=True).reset_index()
# Set Index for fast lookup: .loc[(Species, CO2)]
df_cstars_mean = df_cstars_mean.set_index(['Species', 'GrowthCO2ppm'])

print("✅ Species Means Calculated.")


# =============================================================================
# 3. WORKER FUNCTION (UPDATED LOOKUP)
# =============================================================================

def process_one_group_robust(key, group, df_organelle_mean, df_cstars_mean, fit_params_init, universal_constants, work_dir, mcmc_steps, burn_in):
    species, plant, co2_growth_level = key
    safe_name = f"{species}_{plant}_{co2_growth_level}".replace(" ", "_")

    param_file = os.path.join(work_dir, "temp_results_params", f"{safe_name}.csv")
    curve_file = os.path.join(work_dir, "temp_results_curves", f"{safe_name}_curve.csv")

    # Resume Logic
    if OVERWRITE == False and os.path.exists(param_file):
        return None

    plt.switch_backend('Agg')

    try:
        # --- A. CUSTOM PARAMETER LOGIC ---
        current_fit_params = copy.deepcopy(fit_params_init)

        # (Optional g_bs logic commented out)

        # --- B. PARAMETER LOOKUP (USING MEANS) ---

        # 1. Organelle Params (Lookup by Species + CO2)
        try:
            organelle_row = df_organelle_mean.loc[(species, co2_growth_level)]
        except KeyError:
            # Fallback/Error info
            raise KeyError(f"No organelle mean found for {species} at {co2_growth_level}")

        organelle_params = {
            'f_GDC': float(organelle_row['f_GDC']) / 100.0,
            'f_Rd': float(organelle_row['f_Rd'])  / 100.0,
            'f_rubisco': float(organelle_row['f_rubisco']) / 100.0
        }

        # 2. Rd Params (Lookup by Species + CO2)
        try:
            cstars_row = df_cstars_mean.loc[(species, co2_growth_level)]
        except KeyError:
            raise KeyError(f"No C-star/Rd mean found for {species} at {co2_growth_level}")

        rd_val = float(cstars_row['R_d']) * -1.0
        rd_params = {'R_d': rd_val}

        # Combine
        plant_params_data = organelle_params.copy()
        plant_params_data.update(rd_params)

        # --- C. DATA PREP (Still individual Plant data) ---
        step_data = group.copy()
        step_data['CO2R_Rounded'] = step_data['CO2R'].round(-1)
        averaged_data = step_data.groupby('CO2R_Rounded')[['Photo', 'Ci', 'CO2R']].mean().reset_index().sort_values('CO2R')

        A_data = averaged_data['Photo'].tolist()
        Ci_data = averaged_data['Ci'].tolist()
        Ca_data = averaged_data['CO2R'].tolist()

        if len(A_data) < 3: return None

        # --- D. FIT ---
        fit_output = fit_c2_photosynthesis_model_bayesian(
            A_data=A_data, Ci_data=Ci_data, Ca_data=Ca_data,
            plant_params_data=plant_params_data,
            fit_params_initial=current_fit_params,
            universal_constants_data=universal_constants,
            mcmc_steps=mcmc_steps,
            burn_in=burn_in
        )

        # --- E. EXTRACT RESULTS ---
        fit_res = fit_output.get('fit_results', {})
        stats = fit_res.get('stats', {})

        result_row = {
            'Species': species, 'Plant': plant, 'CO2_Growth': co2_growth_level,
            'Sigma_Estimated': stats.get('sigma_estimated', np.nan),
            'N_Data_Points': stats.get('n_data', np.nan),
            # Save the INPUTS used (which are now means)
            'R_d_Input': rd_val,
            'f_GDC_Input': organelle_params['f_GDC'],
            'f_Rd_Input': organelle_params['f_Rd'],
            'f_rubisco_Input': organelle_params['f_rubisco'],
            'Status': 'SUCCESS',
            'Abs_at_Ambient_Cm': np.nan
        }

        # Parameters
        for param in ['V_cmax', 'g_bs']:
            if param in fit_res:
                result_row[f'{param}_Value'] = fit_res[param]['value']
                result_row[f'{param}_StdErr'] = fit_res[param]['stderr']

        # Glycine Shuttle
        if 'Glycine_Shuttle_Rate' in fit_res:
            result_row['Gly_Shuttle_Rate_Value'] = fit_res['Glycine_Shuttle_Rate']['value']

        if 'Cm_Used_for_GlyRate' in fit_res:
            result_row['Cm_Used_for_GlyRate'] = fit_res['Cm_Used_for_GlyRate']

        # Robust Abs Extraction
        abs_found = False
        possible_keys = ['Abs_at_Ambient_Cm', 'Abs', 'A_bs', 'Abs_gross', 'Assimilation_BS']

        for key in possible_keys:
            if key in fit_res:
                val = fit_res[key]
                if isinstance(val, dict) and 'value' in val:
                    result_row['Abs_at_Ambient_Cm'] = val['value']
                else:
                    result_row['Abs_at_Ambient_Cm'] = val
                abs_found = True
                break

        if not abs_found:
            for key in possible_keys:
                if key in fit_output:
                    result_row['Abs_at_Ambient_Cm'] = fit_output[key]
                    abs_found = True
                    break

        # --- F. CURVE DATA ---
        Cm_model = np.array(fit_output.get('Cm_model_range', fit_output.get('Cm', [])))
        A_model = np.array(fit_output.get('A_model', fit_output.get('A', [])))

        df_curve_data = pd.DataFrame({
            'Species': species,
            'Plant': plant,
            'CO2_Growth': co2_growth_level,
            'Cm_Model': Cm_model,
            'A_Modeled': A_model
        })

        # --- G. PLOTTING ---
        gm_val = universal_constants['g_m']
        As_arr = np.array(A_data)
        Cm_data = np.array(Ci_data) - (As_arr / gm_val)

        vcmax_val = result_row.get('V_cmax_Value', 0)
        gbs_val = result_row.get('g_bs_Value', 0)

        fig, ax = plt.subplots(figsize=(6, 5))
        ax.scatter(Cm_data, As_arr, s=50, c='black', marker='o', edgecolors='black', label='Measured')
        ax.plot(Cm_model, A_model, c='black', linewidth=1.5, label='Model Fit')
        ax.set_xlabel(r"$C_m$ ($\mu$mol mol$^{-1}$)", fontsize=12)
        ax.set_ylabel(r"$A$ ($\mu$mol m$^{-2}$ s$^{-1}$)", fontsize=12)
        ax.tick_params(direction='in', top=True, right=True)

        gbs_max_used = current_fit_params['g_bs']['max']
        stats_text = f"Species: {species}\n$V_{{cmax}}$: {vcmax_val:.1f}\n$g_{{bs}}$: {gbs_val:.4f}\n(Max: {gbs_max_used})"
        ax.text(0.95, 0.05, stats_text, transform=ax.transAxes, fontsize=9, ha='right', va='bottom', bbox=dict(facecolor='white', alpha=0.8))

        plt.tight_layout()
        plt.savefig(os.path.join(work_dir, "Fit_Curves_Individual", f"{safe_name}_Fit.png"), dpi=100)
        plt.close()

        # --- H. SAVE ---
        pd.DataFrame([result_row]).to_csv(param_file, index=False)
        df_curve_data.to_csv(curve_file, index=False)

        return "SUCCESS"

    except Exception as e:
        err_row = {'Species': species, 'Plant': plant, 'CO2_Growth': co2_growth_level, 'Status': f"ERROR: {str(e)}"}
        pd.DataFrame([err_row]).to_csv(param_file, index=False)
        return "ERROR"


# =============================================================================
# 4. EXECUTION
# =============================================================================

grouped_data = df_aci.groupby(['Species', 'Plant', 'CO2'])
print(f"🚀 Starting Parallel Fitting (Using Species Means for Inputs)...")
print(f"   - Temp Params: {temp_param_dir}")

# Pass the MEAN dataframes (df_organelle_mean, df_cstars_mean) instead of the raw ones
tasks = [
    (key, group, df_organelle_mean, df_cstars_mean, DEFAULT_FIT_PARAMS_INITIAL, DEFAULT_UNIVERSAL_CONSTANTS, WORKING_DIR_FILEPATH, MCMC_STEPS, BURN_IN)
    for key, group in grouped_data
]

Parallel(n_jobs=-1, verbose=10)(delayed(process_one_group_robust)(*task) for task in tasks)

# Merge
print("\n🔄 Merging Results...")
param_files = glob.glob(os.path.join(temp_param_dir, "*.csv"))
if param_files:
    df_final = pd.concat((pd.read_csv(f) for f in param_files), ignore_index=True)
    df_final.to_csv(os.path.join(WORKING_DIR_FILEPATH, 'fit_results_species_bayesian_FINAL.csv'), index=False)
    print(f"✅ Params Merged: {len(df_final)} rows")
    if 'Abs_at_Ambient_Cm' in df_final.columns:
        n_abs = df_final['Abs_at_Ambient_Cm'].notna().sum()
        print(f"   (Abs captured for {n_abs} rows)")
    else:
        print("   ⚠️ WARNING: 'Abs_at_Ambient_Cm' column is missing from final merge!")

curve_files = glob.glob(os.path.join(temp_curve_dir, "*.csv"))
if curve_files:
    pd.concat((pd.read_csv(f) for f in curve_files), ignore_index=True).to_csv(os.path.join(WORKING_DIR_FILEPATH, 'fit_results_curves_FINAL.csv'), index=False)
    print("✅ Curves Merged.")

📊 Calculating Species Means for Inputs...
✅ Species Means Calculated.
🚀 Starting Parallel Fitting (Using Species Means for Inputs)...
   - Temp Params: gdrive/My Drive/Colab Notebooks/temp_results_params


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:    2.1s
[Parallel(n_jobs=-1)]: Done   9 tasks      | elapsed:    2.2s
[Parallel(n_jobs=-1)]: Done  16 tasks      | elapsed:    2.2s
[Parallel(n_jobs=-1)]: Done  25 tasks      | elapsed:    2.3s
[Parallel(n_jobs=-1)]: Batch computation too fast (0.19054491546651672s.) Setting batch_size=2.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:    2.3s
[Parallel(n_jobs=-1)]: Done  45 tasks      | elapsed:    2.4s
[Parallel(n_jobs=-1)]: Batch computation too fast (0.07306671142578125s.) Setting batch_size=4.
[Parallel(n_jobs=-1)]: Done  64 tasks      | elapsed:    2.4s
[Parallel(n_jobs=-1)]: Done  85 out of  90 | elapsed:    2.4s remaining:    0.1s
[Parallel(n_jobs=-1)]: Done  90 out of  90 | elapsed:    2.4s finished



🔄 Merging Results...
✅ Params Merged: 90 rows
   (Abs captured for 90 rows)
✅ Curves Merged.


### Sensitivity analysis




In [ ]:
# =============================================================================
# COMPREHENSIVE SENSITIVITY ANALYSIS: f_Rdh × g_m
# =============================================================================
# Tests all combinations of:
#   f_Rdh = [0.0, 0.1, 0.2] (heterotrophic respiration fraction)
#   g_m = [0.3, 0.6, 0.9] (mesophyll conductance)
# Total: 9 combinations × 90 plant groups = 810 fits

import os
import glob
from joblib import Parallel, delayed
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import copy

# =============================================================================
# MODIFIED WORKER FUNCTION FOR COMBINED PARAMETER TESTING
# =============================================================================

def process_one_group_combined_test(key, group, df_organelle_mean, df_cstars_mean,
                                     fit_params_init, universal_constants,
                                     work_dir, mcmc_steps, burn_in,
                                     temp_param_dir_custom, temp_curve_dir_custom, fig_dir_custom):
    """
    Worker function that accepts custom output directories.
    Tests combinations of f_Rdh and g_m.
    """
    species, plant, co2_growth_level = key
    safe_name = f"{species}_{plant}_{co2_growth_level}".replace(" ", "_")

    param_file = os.path.join(temp_param_dir_custom, f"{safe_name}.csv")
    curve_file = os.path.join(temp_curve_dir_custom, f"{safe_name}_curve.csv")

    # Resume Logic
    if OVERWRITE == False and os.path.exists(param_file):
        return None

    plt.switch_backend('Agg')

    try:
        # --- A. CUSTOM PARAMETER LOGIC ---
        current_fit_params = copy.deepcopy(fit_params_init)

        # --- B. PARAMETER LOOKUP (USING MEANS) ---

        # 1. Organelle Params
        try:
            organelle_row = df_organelle_mean.loc[(species, co2_growth_level)]
        except KeyError:
            raise KeyError(f"No organelle mean found for {species} at {co2_growth_level}")

        organelle_params = {
            'f_GDC': float(organelle_row['f_GDC']) / 100.0,
            'f_Rd': float(organelle_row['f_Rd']) / 100.0,
            'f_rubisco': float(organelle_row['f_rubisco']) / 100.0
        }

        # 2. Rd Params
        try:
            cstars_row = df_cstars_mean.loc[(species, co2_growth_level)]
        except KeyError:
            raise KeyError(f"No C-star/Rd mean found for {species} at {co2_growth_level}")

        rd_val = float(cstars_row['R_d']) * -1.0
        rd_params = {'R_d': rd_val}

        # Combine
        plant_params_data = organelle_params.copy()
        plant_params_data.update(rd_params)

        # --- C. DATA PREP ---
        step_data = group.copy()
        step_data['CO2R_Rounded'] = step_data['CO2R'].round(-1)
        averaged_data = step_data.groupby('CO2R_Rounded')[['Photo', 'Ci', 'CO2R']].mean().reset_index().sort_values('CO2R')

        A_data = averaged_data['Photo'].tolist()
        Ci_data = averaged_data['Ci'].tolist()
        Ca_data = averaged_data['CO2R'].tolist()

        if len(A_data) < 3:
            return None

        # --- D. FIT ---
        fit_output = fit_c2_photosynthesis_model_bayesian(
            A_data=A_data, Ci_data=Ci_data, Ca_data=Ca_data,
            plant_params_data=plant_params_data,
            fit_params_initial=current_fit_params,
            universal_constants_data=universal_constants,
            mcmc_steps=mcmc_steps,
            burn_in=burn_in
        )

        # --- E. EXTRACT RESULTS ---
        fit_res = fit_output.get('fit_results', {})
        stats = fit_res.get('stats', {})

        result_row = {
            'Species': species, 'Plant': plant, 'CO2_Growth': co2_growth_level,
            'Sigma_Estimated': stats.get('sigma_estimated', np.nan),
            'N_Data_Points': stats.get('n_data', np.nan),
            'R_d_Input': rd_val,
            'f_GDC_Input': organelle_params['f_GDC'],
            'f_Rd_Input': organelle_params['f_Rd'],
            'f_rubisco_Input': organelle_params['f_rubisco'],
            'g_m_used': universal_constants['g_m'],
            'f_Rdh_used': universal_constants['f_Rdh'],
            'Status': 'SUCCESS',
            'Abs_at_Ambient_Cm': np.nan
        }

        # Parameters
        for param in ['V_cmax', 'g_bs']:
            if param in fit_res:
                result_row[f'{param}_Value'] = fit_res[param]['value']
                result_row[f'{param}_StdErr'] = fit_res[param]['stderr']

        # Glycine Shuttle
        if 'Glycine_Shuttle_Rate' in fit_res:
            result_row['Gly_Shuttle_Rate_Value'] = fit_res['Glycine_Shuttle_Rate']['value']

        if 'Cm_Used_for_GlyRate' in fit_res:
            result_row['Cm_Used_for_GlyRate'] = fit_res['Cm_Used_for_GlyRate']

        # Robust Abs Extraction
        abs_found = False
        possible_keys = ['Abs_at_Ambient_Cm', 'Abs', 'A_bs', 'Abs_gross', 'Assimilation_BS']

        for abs_key in possible_keys:
            if abs_key in fit_res:
                val = fit_res[abs_key]
                if isinstance(val, dict) and 'value' in val:
                    result_row['Abs_at_Ambient_Cm'] = val['value']
                else:
                    result_row['Abs_at_Ambient_Cm'] = val
                abs_found = True
                break

        if not abs_found:
            for abs_key in possible_keys:
                if abs_key in fit_output:
                    result_row['Abs_at_Ambient_Cm'] = fit_output[abs_key]
                    abs_found = True
                    break

        # --- F. CURVE DATA ---
        Cm_model = np.array(fit_output.get('Cm_model_range', fit_output.get('Cm', [])))
        A_model = np.array(fit_output.get('A_model', fit_output.get('A', [])))

        df_curve_data = pd.DataFrame({
            'Species': species,
            'Plant': plant,
            'CO2_Growth': co2_growth_level,
            'Cm_Model': Cm_model,
            'A_Modeled': A_model
        })

        # --- G. PLOTTING ---
        gm_val = universal_constants['g_m']
        f_Rdh_val = universal_constants['f_Rdh']
        As_arr = np.array(A_data)
        Cm_data = np.array(Ci_data) - (As_arr / gm_val)

        vcmax_val = result_row.get('V_cmax_Value', 0)
        gbs_val = result_row.get('g_bs_Value', 0)

        fig, ax = plt.subplots(figsize=(6, 5))
        ax.scatter(Cm_data, As_arr, s=50, c='black', marker='o', edgecolors='black', label='Measured')
        ax.plot(Cm_model, A_model, c='black', linewidth=1.5, label='Model Fit')
        ax.set_xlabel(r"$C_m$ ($\mu$mol mol$^{-1}$)", fontsize=12)
        ax.set_ylabel(r"$A$ ($\mu$mol m$^{-2}$ s$^{-1}$)", fontsize=12)
        ax.tick_params(direction='in', top=True, right=True)

        gbs_max_used = current_fit_params['g_bs']['max']
        stats_text = f"Species: {species}\n$V_{{cmax}}$: {vcmax_val:.1f}\n$g_{{bs}}$: {gbs_val:.4f}\n$g_{{m}}$: {gm_val:.1f}\n$f_{{Rdh}}$: {f_Rdh_val:.2f}"
        ax.text(0.95, 0.05, stats_text, transform=ax.transAxes, fontsize=9, ha='right', va='bottom',
                bbox=dict(facecolor='white', alpha=0.8))

        plt.tight_layout()
        plt.savefig(os.path.join(fig_dir_custom, f"{safe_name}_Fit.png"), dpi=100)
        plt.close()

        # --- H. SAVE ---
        pd.DataFrame([result_row]).to_csv(param_file, index=False)
        df_curve_data.to_csv(curve_file, index=False)

        return "SUCCESS"

    except Exception as e:
        err_row = {
            'Species': species,
            'Plant': plant,
            'CO2_Growth': co2_growth_level,
            'g_m_used': universal_constants['g_m'],
            'f_Rdh_used': universal_constants['f_Rdh'],
            'Status': f"ERROR: {str(e)}"
        }
        pd.DataFrame([err_row]).to_csv(param_file, index=False)
        return "ERROR"


# =============================================================================
# MAIN NESTED LOOP: f_Rdh × g_m
# =============================================================================

# Define parameter values to test
f_Rdh_values = [0.0, 0.1, 0.2]
gm_values = [0.3, 0.6, 0.9]

# Store all results for comprehensive comparison
all_results_combined = []
combination_summary = []

# Nested loop through all combinations
combination_num = 0
total_combinations = len(f_Rdh_values) * len(gm_values)

for f_Rdh_test in f_Rdh_values:
    for gm_test in gm_values:
        combination_num += 1

        print(f"\n{'='*80}")
        print(f"🔬 COMBINATION {combination_num}/{total_combinations}: f_Rdh = {f_Rdh_test}, g_m = {gm_test}")
        print(f"{'='*80}\n")

        # Update universal constants with current combination
        UNIVERSAL_CONSTANTS_COMBO = DEFAULT_UNIVERSAL_CONSTANTS.copy()
        UNIVERSAL_CONSTANTS_COMBO['f_Rdh'] = f_Rdh_test
        UNIVERSAL_CONSTANTS_COMBO['g_m'] = gm_test

        # Create output directories specific to this combination
        combo_suffix = f"_fRdh_{str(f_Rdh_test).replace('.', 'p')}_gm_{str(gm_test).replace('.', 'p')}"
        temp_param_dir_combo = os.path.join(WORKING_DIR_FILEPATH, f"temp_params{combo_suffix}")
        temp_curve_dir_combo = os.path.join(WORKING_DIR_FILEPATH, f"temp_curves{combo_suffix}")
        fit_curves_dir_combo = os.path.join(WORKING_DIR_FILEPATH, f"Fit_Curves_Individual{combo_suffix}")

        os.makedirs(temp_param_dir_combo, exist_ok=True)
        os.makedirs(temp_curve_dir_combo, exist_ok=True)
        os.makedirs(fit_curves_dir_combo, exist_ok=True)

        # Prepare tasks
        grouped_data = df_aci.groupby(['Species', 'Plant', 'CO2'])
        tasks_combo = [
            (key, group, df_organelle_mean, df_cstars_mean,
             DEFAULT_FIT_PARAMS_INITIAL, UNIVERSAL_CONSTANTS_COMBO,
             WORKING_DIR_FILEPATH, MCMC_STEPS, BURN_IN,
             temp_param_dir_combo, temp_curve_dir_combo, fit_curves_dir_combo)
            for key, group in grouped_data
        ]

        print(f"🚀 Starting Parallel Fitting...")
        print(f"   - Number of groups to fit: {len(tasks_combo)}")
        print(f"   - Temp Params: {temp_param_dir_combo}\n")

        # Run parallel fitting
        Parallel(n_jobs=-1, verbose=10)(
            delayed(process_one_group_combined_test)(*task) for task in tasks_combo
        )

        # Merge results for this combination
        print(f"\n🔄 Merging Results for f_Rdh={f_Rdh_test}, g_m={gm_test}...")

        param_files_combo = glob.glob(os.path.join(temp_param_dir_combo, "*.csv"))
        if param_files_combo:
            df_final_combo = pd.concat((pd.read_csv(f) for f in param_files_combo), ignore_index=True)
            output_csv = os.path.join(WORKING_DIR_FILEPATH,
                                     f'fit_results_fRdh_{str(f_Rdh_test).replace(".", "p")}_gm_{str(gm_test).replace(".", "p")}_FINAL.csv')
            df_final_combo.to_csv(output_csv, index=False)

            # Store for comparison
            all_results_combined.append(df_final_combo.copy())

            # Count successes and errors
            n_success = (df_final_combo['Status'] == 'SUCCESS').sum()
            n_error = (df_final_combo['Status'] != 'SUCCESS').sum()

            print(f"✅ Params Merged: {len(df_final_combo)} rows")
            print(f"   SUCCESS: {n_success}, ERRORS: {n_error}")
            print(f"   Saved to: {output_csv}")

            # Summary statistics for successful fits
            df_success = df_final_combo[df_final_combo['Status'] == 'SUCCESS']
            if len(df_success) > 0:
                summary_row = {
                    'f_Rdh': f_Rdh_test,
                    'g_m': gm_test,
                    'n_success': n_success,
                    'n_error': n_error,
                    'g_bs_mean': df_success['g_bs_Value'].mean() if 'g_bs_Value' in df_success.columns else np.nan,
                    'g_bs_std': df_success['g_bs_Value'].std() if 'g_bs_Value' in df_success.columns else np.nan,
                    'V_cmax_mean': df_success['V_cmax_Value'].mean() if 'V_cmax_Value' in df_success.columns else np.nan,
                    'V_cmax_std': df_success['V_cmax_Value'].std() if 'V_cmax_Value' in df_success.columns else np.nan,
                    'Abs_mean': df_success['Abs_at_Ambient_Cm'].mean() if 'Abs_at_Ambient_Cm' in df_success.columns else np.nan,
                    'Abs_std': df_success['Abs_at_Ambient_Cm'].std() if 'Abs_at_Ambient_Cm' in df_success.columns else np.nan
                }
                combination_summary.append(summary_row)

                print(f"\n   Summary for {n_success} successful fits:")
                print(f"      g_bs: mean={summary_row['g_bs_mean']:.4f}, std={summary_row['g_bs_std']:.4f}")
                print(f"      V_cmax: mean={summary_row['V_cmax_mean']:.2f}, std={summary_row['V_cmax_std']:.2f}")
            else:
                combination_summary.append({
                    'f_Rdh': f_Rdh_test,
                    'g_m': gm_test,
                    'n_success': 0,
                    'n_error': n_error,
                    'g_bs_mean': np.nan,
                    'g_bs_std': np.nan,
                    'V_cmax_mean': np.nan,
                    'V_cmax_std': np.nan,
                    'Abs_mean': np.nan,
                    'Abs_std': np.nan
                })

        # Save curves
        curve_files_combo = glob.glob(os.path.join(temp_curve_dir_combo, "*.csv"))
        if curve_files_combo:
            df_curves_combo = pd.concat((pd.read_csv(f) for f in curve_files_combo), ignore_index=True)
            output_csv_curves = os.path.join(WORKING_DIR_FILEPATH,
                                            f'fit_results_curves_fRdh_{str(f_Rdh_test).replace(".", "p")}_gm_{str(gm_test).replace(".", "p")}_FINAL.csv')
            df_curves_combo.to_csv(output_csv_curves, index=False)
            print(f"✅ Curves Merged and saved")


# =============================================================================
# COMPREHENSIVE ANALYSIS AND VISUALIZATION
# =============================================================================

if all_results_combined:
    print(f"\n{'='*80}")
    print("📊 COMPREHENSIVE ANALYSIS: ALL COMBINATIONS")
    print(f"{'='*80}\n")

    # Combine all results
    df_all_combined = pd.concat(all_results_combined, ignore_index=True)
    output_combined = os.path.join(WORKING_DIR_FILEPATH, 'fit_results_ALL_COMBINATIONS.csv')
    df_all_combined.to_csv(output_combined, index=False)

    df_success_all = df_all_combined[df_all_combined['Status'] == 'SUCCESS']
    n_total_success = len(df_success_all)
    n_total = len(df_all_combined)

    print(f"✅ Combined results saved to: {output_combined}")
    print(f"   Total fits attempted: {n_total}")
    print(f"   Successful fits: {n_total_success} ({100*n_total_success/n_total:.1f}%)")
    print(f"   Failed fits: {n_total - n_total_success} ({100*(n_total - n_total_success)/n_total:.1f}%)")

    # Save combination summary
    df_summary = pd.DataFrame(combination_summary)
    summary_file = os.path.join(WORKING_DIR_FILEPATH, 'combination_summary.csv')
    df_summary.to_csv(summary_file, index=False)
    print(f"\n✅ Combination summary saved to: {summary_file}")

    # Display summary table
    print("\n📈 SUMMARY BY COMBINATION:")
    print("-" * 100)
    print(df_summary.to_string(index=False))

    if n_total_success > 0:
        # Overall statistics
        print("\n📊 OVERALL STATISTICS (SUCCESSFUL FITS ONLY):")
        print("-" * 80)
        overall_stats = df_success_all.groupby(['f_Rdh_used', 'g_m_used']).agg({
            'g_bs_Value': ['mean', 'std', 'count'],
            'V_cmax_Value': ['mean', 'std'],
            'Abs_at_Ambient_Cm': ['mean', 'std']
        }).round(4)
        print(overall_stats)

        # =================================================================
        # CREATE HEATMAPS
        # =================================================================
        print("\n📊 Creating interaction heatmaps...")

        # Prepare pivot tables for heatmaps
        pivot_gbs = df_success_all.pivot_table(values='g_bs_Value',
                                                index='f_Rdh_used',
                                                columns='g_m_used',
                                                aggfunc='mean')

        pivot_vcmax = df_success_all.pivot_table(values='V_cmax_Value',
                                                  index='f_Rdh_used',
                                                  columns='g_m_used',
                                                  aggfunc='mean')

        pivot_abs = df_success_all.pivot_table(values='Abs_at_Ambient_Cm',
                                                index='f_Rdh_used',
                                                columns='g_m_used',
                                                aggfunc='mean')

        pivot_count = df_success_all.pivot_table(values='g_bs_Value',
                                                  index='f_Rdh_used',
                                                  columns='g_m_used',
                                                  aggfunc='count')

        # Create 2x2 subplot of heatmaps
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(14, 12))

        # g_bs heatmap
        sns.heatmap(pivot_gbs, annot=True, fmt='.4f', cmap='viridis', ax=ax1, cbar_kws={'label': 'g_bs'})
        ax1.set_title('Mean g_bs by f_Rdh and g_m', fontsize=14, fontweight='bold')
        ax1.set_xlabel('g_m', fontsize=12)
        ax1.set_ylabel('f_Rdh', fontsize=12)

        # V_cmax heatmap
        sns.heatmap(pivot_vcmax, annot=True, fmt='.1f', cmap='plasma', ax=ax2, cbar_kws={'label': 'V_cmax'})
        ax2.set_title('Mean V_cmax by f_Rdh and g_m', fontsize=14, fontweight='bold')
        ax2.set_xlabel('g_m', fontsize=12)
        ax2.set_ylabel('f_Rdh', fontsize=12)

        # A_bs heatmap
        sns.heatmap(pivot_abs, annot=True, fmt='.2f', cmap='coolwarm', ax=ax3, cbar_kws={'label': 'A_bs'})
        ax3.set_title('Mean A_bs by f_Rdh and g_m', fontsize=14, fontweight='bold')
        ax3.set_xlabel('g_m', fontsize=12)
        ax3.set_ylabel('f_Rdh', fontsize=12)

        # Success count heatmap
        sns.heatmap(pivot_count, annot=True, fmt='.0f', cmap='Greens', ax=ax4, cbar_kws={'label': 'Count'})
        ax4.set_title('Number of Successful Fits', fontsize=14, fontweight='bold')
        ax4.set_xlabel('g_m', fontsize=12)
        ax4.set_ylabel('f_Rdh', fontsize=12)

        plt.tight_layout()
        heatmap_file = os.path.join(WORKING_DIR_FILEPATH, 'interaction_heatmaps.png')
        plt.savefig(heatmap_file, dpi=150)
        plt.close()
        print(f"   Saved heatmaps: {heatmap_file}")

        # =================================================================
        # CREATE INTERACTION PLOTS
        # =================================================================
        print("📊 Creating interaction line plots...")

        fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))

        # g_bs interaction plot
        for f_Rdh_val in f_Rdh_values:
            subset = df_success_all[df_success_all['f_Rdh_used'] == f_Rdh_val]
            gm_means = subset.groupby('g_m_used')['g_bs_Value'].mean()
            ax1.plot(gm_means.index, gm_means.values, marker='o', label=f'f_Rdh={f_Rdh_val}', linewidth=2, markersize=8)
        ax1.set_xlabel('g_m', fontsize=12)
        ax1.set_ylabel('Mean g_bs', fontsize=12)
        ax1.set_title('g_bs: Interaction between f_Rdh and g_m', fontsize=14, fontweight='bold')
        ax1.legend()
        ax1.grid(True, alpha=0.3)

        # V_cmax interaction plot
        for f_Rdh_val in f_Rdh_values:
            subset = df_success_all[df_success_all['f_Rdh_used'] == f_Rdh_val]
            gm_means = subset.groupby('g_m_used')['V_cmax_Value'].mean()
            ax2.plot(gm_means.index, gm_means.values, marker='o', label=f'f_Rdh={f_Rdh_val}', linewidth=2, markersize=8)
        ax2.set_xlabel('g_m', fontsize=12)
        ax2.set_ylabel('Mean V_cmax', fontsize=12)
        ax2.set_title('V_cmax: Interaction between f_Rdh and g_m', fontsize=14, fontweight='bold')
        ax2.legend()
        ax2.grid(True, alpha=0.3)

        # A_bs interaction plot
        for f_Rdh_val in f_Rdh_values:
            subset = df_success_all[df_success_all['f_Rdh_used'] == f_Rdh_val]
            gm_means = subset.groupby('g_m_used')['Abs_at_Ambient_Cm'].mean()
            ax3.plot(gm_means.index, gm_means.values, marker='o', label=f'f_Rdh={f_Rdh_val}', linewidth=2, markersize=8)
        ax3.set_xlabel('g_m', fontsize=12)
        ax3.set_ylabel('Mean A_bs', fontsize=12)
        ax3.set_title('A_bs: Interaction between f_Rdh and g_m', fontsize=14, fontweight='bold')
        ax3.legend()
        ax3.grid(True, alpha=0.3)

        plt.tight_layout()
        interaction_file = os.path.join(WORKING_DIR_FILEPATH, 'interaction_plots.png')
        plt.savefig(interaction_file, dpi=150)
        plt.close()
        print(f"   Saved interaction plots: {interaction_file}")

    else:
        print("\n⚠️ No successful fits found across all combinations")

print(f"\n{'='*80}")
print("🎉 COMPREHENSIVE SENSITIVITY ANALYSIS COMPLETE!")
print(f"{'='*80}")
print(f"\nTested {total_combinations} combinations of f_Rdh × g_m")
print(f"Total fits attempted: {len(all_results_combined[0]) * total_combinations if all_results_combined else 0}")
print("\nResults saved:")
print(f"  - fit_results_ALL_COMBINATIONS.csv (master file)")
print(f"  - combination_summary.csv (summary statistics)")
if n_total_success > 0:
    print(f"  - interaction_heatmaps.png (4-panel heatmap)")
    print(f"  - interaction_plots.png (interaction line plots)")
print(f"\nIndividual combination files:")
for f_Rdh_test in f_Rdh_values:
    for gm_test in gm_values:
        combo_str = f"fRdh_{str(f_Rdh_test).replace('.', 'p')}_gm_{str(gm_test).replace('.', 'p')}"
        print(f"  - fit_results_{combo_str}_FINAL.csv")


🔬 COMBINATION 1/9: f_Rdh = 0.0, g_m = 0.3

🚀 Starting Parallel Fitting...
   - Number of groups to fit: 90
   - Temp Params: gdrive/My Drive/Colab Notebooks/temp_params_fRdh_0p0_gm_0p3



[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:  1.6min
[Parallel(n_jobs=-1)]: Done   9 tasks      | elapsed:  3.1min
[Parallel(n_jobs=-1)]: Done  16 tasks      | elapsed:  3.2min
[Parallel(n_jobs=-1)]: Done  25 tasks      | elapsed:  6.2min
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:  7.8min
[Parallel(n_jobs=-1)]: Done  45 tasks      | elapsed:  9.4min
[Parallel(n_jobs=-1)]: Done  56 tasks      | elapsed: 11.1min
[Parallel(n_jobs=-1)]: Done  69 tasks      | elapsed: 14.1min
[Parallel(n_jobs=-1)]: Done  85 out of  90 | elapsed: 17.0min remaining:   60.0s
[Parallel(n_jobs=-1)]: Done  90 out of  90 | elapsed: 17.8min finished



🔄 Merging Results for f_Rdh=0.0, g_m=0.3...
✅ Params Merged: 90 rows
   SUCCESS: 90, ERRORS: 0
   Saved to: gdrive/My Drive/Colab Notebooks/fit_results_fRdh_0p0_gm_0p3_FINAL.csv

   Summary for 90 successful fits:
      g_bs: mean=0.0531, std=0.0510
      V_cmax: mean=153.44, std=49.62
✅ Curves Merged and saved

🔬 COMBINATION 2/9: f_Rdh = 0.0, g_m = 0.6

🚀 Starting Parallel Fitting...
   - Number of groups to fit: 90
   - Temp Params: gdrive/My Drive/Colab Notebooks/temp_params_fRdh_0p0_gm_0p6



[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:  1.5min
[Parallel(n_jobs=-1)]: Done   9 tasks      | elapsed:  3.0min
[Parallel(n_jobs=-1)]: Done  16 tasks      | elapsed:  3.2min
[Parallel(n_jobs=-1)]: Done  25 tasks      | elapsed:  6.1min
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:  7.7min
[Parallel(n_jobs=-1)]: Done  45 tasks      | elapsed:  9.3min
[Parallel(n_jobs=-1)]: Done  56 tasks      | elapsed: 11.1min
[Parallel(n_jobs=-1)]: Done  69 tasks      | elapsed: 14.0min
[Parallel(n_jobs=-1)]: Done  85 out of  90 | elapsed: 17.1min remaining:  1.0min
[Parallel(n_jobs=-1)]: Done  90 out of  90 | elapsed: 17.8min finished



🔄 Merging Results for f_Rdh=0.0, g_m=0.6...
✅ Params Merged: 90 rows
   SUCCESS: 90, ERRORS: 0
   Saved to: gdrive/My Drive/Colab Notebooks/fit_results_fRdh_0p0_gm_0p6_FINAL.csv

   Summary for 90 successful fits:
      g_bs: mean=0.0503, std=0.0505
      V_cmax: mean=122.54, std=33.43
✅ Curves Merged and saved

🔬 COMBINATION 3/9: f_Rdh = 0.0, g_m = 0.9

🚀 Starting Parallel Fitting...
   - Number of groups to fit: 90
   - Temp Params: gdrive/My Drive/Colab Notebooks/temp_params_fRdh_0p0_gm_0p9



...


🔄 Merging Results for f_Rdh=0.2, g_m=0.3...
✅ Params Merged: 90 rows
   SUCCESS: 90, ERRORS: 0
   Saved to: gdrive/My Drive/Colab Notebooks/fit_results_fRdh_0p2_gm_0p3_FINAL.csv

   Summary for 90 successful fits:
      g_bs: mean=0.0552, std=0.0518
      V_cmax: mean=151.56, std=49.42
✅ Curves Merged and saved

🔬 COMBINATION 8/9: f_Rdh = 0.2, g_m = 0.6

🚀 Starting Parallel Fitting...
   - Number of groups to fit: 90
   - Temp Params: gdrive/My Drive/Colab Notebooks/temp_params_fRdh_0p2_gm_0p6



[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:  1.6min
[Parallel(n_jobs=-1)]: Done   9 tasks      | elapsed:  3.1min
[Parallel(n_jobs=-1)]: Done  16 tasks      | elapsed:  3.2min
[Parallel(n_jobs=-1)]: Done  25 tasks      | elapsed:  6.2min
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:  7.9min
[Parallel(n_jobs=-1)]: Done  45 tasks      | elapsed:  9.5min
[Parallel(n_jobs=-1)]: Done  56 tasks      | elapsed: 11.2min
[Parallel(n_jobs=-1)]: Done  69 tasks      | elapsed: 14.2min
[Parallel(n_jobs=-1)]: Done  85 out of  90 | elapsed: 17.3min remaining:  1.0min
[Parallel(n_jobs=-1)]: Done  90 out of  90 | elapsed: 18.1min finished



🔄 Merging Results for f_Rdh=0.2, g_m=0.6...
✅ Params Merged: 90 rows
   SUCCESS: 90, ERRORS: 0
   Saved to: gdrive/My Drive/Colab Notebooks/fit_results_fRdh_0p2_gm_0p6_FINAL.csv

   Summary for 90 successful fits:
      g_bs: mean=0.0547, std=0.0529
      V_cmax: mean=120.84, std=33.03
✅ Curves Merged and saved

🔬 COMBINATION 9/9: f_Rdh = 0.2, g_m = 0.9

🚀 Starting Parallel Fitting...
   - Number of groups to fit: 90
   - Temp Params: gdrive/My Drive/Colab Notebooks/temp_params_fRdh_0p2_gm_0p9



[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:  1.6min
[Parallel(n_jobs=-1)]: Done   9 tasks      | elapsed:  3.0min
[Parallel(n_jobs=-1)]: Done  16 tasks      | elapsed:  3.2min
[Parallel(n_jobs=-1)]: Done  25 tasks      | elapsed:  6.3min
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:  8.1min
[Parallel(n_jobs=-1)]: Done  45 tasks      | elapsed:  9.8min
[Parallel(n_jobs=-1)]: Done  56 tasks      | elapsed: 11.4min
[Parallel(n_jobs=-1)]: Done  69 tasks      | elapsed: 14.5min
[Parallel(n_jobs=-1)]: Done  85 out of  90 | elapsed: 17.6min remaining:  1.0min
[Parallel(n_jobs=-1)]: Done  90 out of  90 | elapsed: 18.4min finished



🔄 Merging Results for f_Rdh=0.2, g_m=0.9...
✅ Params Merged: 90 rows
   SUCCESS: 90, ERRORS: 0
   Saved to: gdrive/My Drive/Colab Notebooks/fit_results_fRdh_0p2_gm_0p9_FINAL.csv

   Summary for 90 successful fits:
      g_bs: mean=0.0543, std=0.0529
      V_cmax: mean=113.32, std=29.02
✅ Curves Merged and saved

📊 COMPREHENSIVE ANALYSIS: ALL COMBINATIONS

✅ Combined results saved to: gdrive/My Drive/Colab Notebooks/fit_results_ALL_COMBINATIONS.csv
   Total fits attempted: 810
   Successful fits: 810 (100.0%)
   Failed fits: 0 (0.0%)

✅ Combination summary saved to: gdrive/My Drive/Colab Notebooks/combination_summary.csv

📈 SUMMARY BY COMBINATION:
----------------------------------------------------------------------------------------------------
 f_Rdh  g_m  n_success  n_error  g_bs_mean  g_bs_std  V_cmax_mean  V_cmax_std  Abs_mean  Abs_std
   0.0  0.3         90        0   0.053064  0.050967   153.443098   49.622218  3.571902 3.007222
   0.0  0.6         90        0   0.050305  0.050

### Comparisons of CO2 treatment effects

In [ ]:
# =============================================================================
# CO2 TREATMENT EFFECT ANALYSIS: How do 200 vs 800 ppm differences depend on g_m and f_Rdh?
# =============================================================================
# This chunk analyzes whether the physiological responses to elevated CO2
# (800 ppm vs 200 ppm) are sensitive to model assumptions about g_m and f_Rdh

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# =============================================================================
# 1. LOAD THE COMPREHENSIVE RESULTS
# =============================================================================

print("="*80)
print("📊 ANALYZING CO2 TREATMENT EFFECTS ACROSS PARAMETER COMBINATIONS")
print("="*80)

# Load the master combined results file
results_file = os.path.join(WORKING_DIR_FILEPATH, 'fit_results_ALL_COMBINATIONS.csv')

if not os.path.exists(results_file):
    print(f"\n❌ ERROR: Could not find {results_file}")
    print("Please run the comprehensive sensitivity analysis first!")
else:
    df_all = pd.read_csv(results_file)

    # Filter for successful fits only
    df_success = df_all[df_all['Status'] == 'SUCCESS'].copy()

    # Define global parameter values for use throughout analysis
    gm_values = sorted(df_success['g_m_used'].unique())
    f_rdh_values = sorted(df_success['f_Rdh_used'].unique())
    species_list = sorted(df_success['Species'].unique())

    print(f"\n✅ Loaded {len(df_all)} total fits")
    print(f"   Successful fits: {len(df_success)} ({100*len(df_success)/len(df_all):.1f}%)")
    print(f"   g_m values: {gm_values}")
    print(f"   f_Rdh values: {f_rdh_values}")
    print(f"   Species: {len(species_list)}")

    # =============================================================================
    # 2. CALCULATE CO2 TREATMENT DIFFERENCES PER SPECIES (POOLING PLANTS)
    # =============================================================================

    print("\n📈 Calculating CO2 treatment differences per species (pooling plants)...")

    # Parameters to analyze
    params_to_analyze = ['g_bs_Value', 'V_cmax_Value', 'Abs_at_Ambient_Cm', 'Gly_Shuttle_Rate_Value']

    # Create a list to store difference data
    diff_results = []

    # Group by Species, g_m_used, f_Rdh_used, CO2_Growth
    # First, calculate mean values across plants for each species × CO2 × parameter combination
    species_means = df_success.groupby(['Species', 'g_m_used', 'f_Rdh_used', 'CO2_Growth'])[params_to_analyze].mean().reset_index()

    # Now calculate differences per species for each parameter combination
    grouped = species_means.groupby(['Species', 'g_m_used', 'f_Rdh_used'])

    for (species, gm, f_rdh), group in grouped:
        # Check if we have both CO2 treatments
        co2_levels = group['CO2_Growth'].unique()

        if 200 in co2_levels and 800 in co2_levels:
            # Get 200 ppm data (mean across plants)
            data_200 = group[group['CO2_Growth'] == 200].iloc[0]
            # Get 800 ppm data (mean across plants)
            data_800 = group[group['CO2_Growth'] == 800].iloc[0]

            diff_row = {
                'Species': species,
                'g_m_used': gm,
                'f_Rdh_used': f_rdh
            }

            # Calculate differences and percent changes for each parameter
            for param in params_to_analyze:
                if param in data_200.index and param in data_800.index:
                    val_200 = data_200[param]
                    val_800 = data_800[param]

                    if pd.notna(val_200) and pd.notna(val_800):
                        # Absolute difference (800 - 200)
                        diff_row[f'{param}_diff'] = val_800 - val_200

                        # Percent change: (800 - 200) / 200 * 100
                        if val_200 != 0:
                            diff_row[f'{param}_pct_change'] = ((val_800 - val_200) / abs(val_200)) * 100
                        else:
                            diff_row[f'{param}_pct_change'] = np.nan

                        # Relative difference: (800 - 200) / mean(200, 800) * 100
                        # This is symmetric and bounded, useful when comparing increases vs decreases
                        mean_val = (val_200 + val_800) / 2
                        if mean_val != 0:
                            diff_row[f'{param}_rel_diff'] = ((val_800 - val_200) / mean_val) * 100
                        else:
                            diff_row[f'{param}_rel_diff'] = np.nan

                        # Store individual values too
                        diff_row[f'{param}_200ppm'] = val_200
                        diff_row[f'{param}_800ppm'] = val_800

            diff_results.append(diff_row)

    df_diff = pd.DataFrame(diff_results)

    print(f"   Calculated differences for {len(df_diff)} species × parameter combinations")
    print(f"   Species: {df_diff['Species'].nunique()}")
    print(f"   Parameter combinations (g_m × f_Rdh): {len(gm_values) * len(f_rdh_values)}")

    # Save the differences dataset
    diff_file = os.path.join(WORKING_DIR_FILEPATH, 'CO2_treatment_differences.csv')
    df_diff.to_csv(diff_file, index=False)
    print(f"\n✅ Saved CO2 treatment differences to: {diff_file}")

    # =============================================================================
    # 3. SUMMARY STATISTICS BY PARAMETER COMBINATION
    # =============================================================================

    print("\n📊 Summary statistics by g_m and f_Rdh combination:")
    print("-"*80)

    # Calculate mean differences for each combination
    summary_stats = df_diff.groupby(['g_m_used', 'f_Rdh_used']).agg({
        'g_bs_Value_diff': ['mean', 'std', 'count'],
        'V_cmax_Value_diff': ['mean', 'std'],
        'Abs_at_Ambient_Cm_diff': ['mean', 'std'],
        'g_bs_Value_pct_change': 'mean',
        'V_cmax_Value_pct_change': 'mean',
        'Abs_at_Ambient_Cm_pct_change': 'mean',
        'g_bs_Value_rel_diff': 'mean',
        'V_cmax_Value_rel_diff': 'mean',
        'Abs_at_Ambient_Cm_rel_diff': 'mean'
    }).round(4)

    print(summary_stats)

    # Save summary
    summary_file = os.path.join(WORKING_DIR_FILEPATH, 'CO2_treatment_summary_by_params.csv')
    summary_stats.to_csv(summary_file)
    print(f"\n✅ Saved summary to: {summary_file}")

    # =============================================================================
    # 4. STATISTICAL TESTING: ANOVA
    # =============================================================================

    print("\n🔬 Testing whether CO2 responses differ by parameter assumptions...")

    stats_results = []

    for param in ['g_bs_Value_diff', 'V_cmax_Value_diff', 'Abs_at_Ambient_Cm_diff']:
        if param in df_diff.columns:
            # Test effect of g_m on CO2 response
            gm_groups = [df_diff[df_diff['g_m_used'] == gm][param].dropna() for gm in gm_values]
            if len(gm_groups) > 1 and all(len(g) > 0 for g in gm_groups):
                f_stat, p_val = stats.f_oneway(*gm_groups)
                stats_results.append({
                    'Parameter': param,
                    'Factor': 'g_m',
                    'F_statistic': f_stat,
                    'p_value': p_val,
                    'Significant': 'Yes' if p_val < 0.05 else 'No'
                })

            # Test effect of f_Rdh on CO2 response
            frdh_groups = [df_diff[df_diff['f_Rdh_used'] == frdh][param].dropna() for frdh in f_rdh_values]
            if len(frdh_groups) > 1 and all(len(g) > 0 for g in frdh_groups):
                f_stat, p_val = stats.f_oneway(*frdh_groups)
                stats_results.append({
                    'Parameter': param,
                    'Factor': 'f_Rdh',
                    'F_statistic': f_stat,
                    'p_value': p_val,
                    'Significant': 'Yes' if p_val < 0.05 else 'No'
                })

    if stats_results:
        df_stats = pd.DataFrame(stats_results)
        print("\nANOVA Results:")
        print("-"*80)
        print(df_stats.to_string(index=False))

        stats_file = os.path.join(WORKING_DIR_FILEPATH, 'CO2_response_ANOVA.csv')
        df_stats.to_csv(stats_file, index=False)
        print(f"\n✅ Saved ANOVA results to: {stats_file}")
    else:
        print("   No ANOVA results generated")



    # =============================================================================
    # 5. VISUALIZATION: HEATMAPS OF CO2 RESPONSES
    # =============================================================================

    print("\n📊 Creating heatmaps of CO2 treatment effects...")

    # Create pivot tables for heatmaps
    pivot_gbs_diff = df_diff.pivot_table(values='g_bs_Value_diff',
                                          index='f_Rdh_used',
                                          columns='g_m_used',
                                          aggfunc='mean')

    pivot_vcmax_diff = df_diff.pivot_table(values='V_cmax_Value_diff',
                                            index='f_Rdh_used',
                                            columns='g_m_used',
                                            aggfunc='mean')

    pivot_abs_diff = df_diff.pivot_table(values='Abs_at_Ambient_Cm_diff',
                                          index='f_Rdh_used',
                                          columns='g_m_used',
                                          aggfunc='mean')

    # Percent changes
    pivot_gbs_pct = df_diff.pivot_table(values='g_bs_Value_pct_change',
                                         index='f_Rdh_used',
                                         columns='g_m_used',
                                         aggfunc='mean')

    pivot_vcmax_pct = df_diff.pivot_table(values='V_cmax_Value_pct_change',
                                           index='f_Rdh_used',
                                           columns='g_m_used',
                                           aggfunc='mean')

    pivot_abs_pct = df_diff.pivot_table(values='Abs_at_Ambient_Cm_pct_change',
                                         index='f_Rdh_used',
                                         columns='g_m_used',
                                         aggfunc='mean')

    # Relative differences
    pivot_gbs_rel = df_diff.pivot_table(values='g_bs_Value_rel_diff',
                                         index='f_Rdh_used',
                                         columns='g_m_used',
                                         aggfunc='mean')

    pivot_vcmax_rel = df_diff.pivot_table(values='V_cmax_Value_rel_diff',
                                           index='f_Rdh_used',
                                           columns='g_m_used',
                                           aggfunc='mean')

    pivot_abs_rel = df_diff.pivot_table(values='Abs_at_Ambient_Cm_rel_diff',
                                         index='f_Rdh_used',
                                         columns='g_m_used',
                                         aggfunc='mean')

    # Create figure with 3 rows, 3 columns
    fig, axes = plt.subplots(3, 3, figsize=(18, 16))

    # Row 1: Absolute differences
    sns.heatmap(pivot_gbs_diff, annot=True, fmt='.4f', cmap='RdBu_r', center=0,
                ax=axes[0,0], cbar_kws={'label': 'Δg_bs'})
    axes[0,0].set_title('Δg_bs (800 - 200 ppm)', fontsize=14, fontweight='bold')
    axes[0,0].set_xlabel('g_m', fontsize=12)
    axes[0,0].set_ylabel('f_Rdh', fontsize=12)

    sns.heatmap(pivot_vcmax_diff, annot=True, fmt='.1f', cmap='RdBu_r', center=0,
                ax=axes[0,1], cbar_kws={'label': 'ΔV_cmax'})
    axes[0,1].set_title('ΔV_cmax (800 - 200 ppm)', fontsize=14, fontweight='bold')
    axes[0,1].set_xlabel('g_m', fontsize=12)
    axes[0,1].set_ylabel('f_Rdh', fontsize=12)

    sns.heatmap(pivot_abs_diff, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
                ax=axes[0,2], cbar_kws={'label': 'ΔA_bs'})
    axes[0,2].set_title('ΔA_bs (800 - 200 ppm)', fontsize=14, fontweight='bold')
    axes[0,2].set_xlabel('g_m', fontsize=12)
    axes[0,2].set_ylabel('f_Rdh', fontsize=12)

    # Row 2: Percent changes
    sns.heatmap(pivot_gbs_pct, annot=True, fmt='.1f', cmap='RdBu_r', center=0,
                ax=axes[1,0], cbar_kws={'label': '% change'})
    axes[1,0].set_title('g_bs % change (800 vs 200 ppm)', fontsize=14, fontweight='bold')
    axes[1,0].set_xlabel('g_m', fontsize=12)
    axes[1,0].set_ylabel('f_Rdh', fontsize=12)

    sns.heatmap(pivot_vcmax_pct, annot=True, fmt='.1f', cmap='RdBu_r', center=0,
                ax=axes[1,1], cbar_kws={'label': '% change'})
    axes[1,1].set_title('V_cmax % change (800 vs 200 ppm)', fontsize=14, fontweight='bold')
    axes[1,1].set_xlabel('g_m', fontsize=12)
    axes[1,1].set_ylabel('f_Rdh', fontsize=12)

    sns.heatmap(pivot_abs_pct, annot=True, fmt='.1f', cmap='RdBu_r', center=0,
                ax=axes[1,2], cbar_kws={'label': '% change'})
    axes[1,2].set_title('A_bs % change (800 vs 200 ppm)', fontsize=14, fontweight='bold')
    axes[1,2].set_xlabel('g_m', fontsize=12)
    axes[1,2].set_ylabel('f_Rdh', fontsize=12)

    # Row 3: Relative differences (symmetric)
    sns.heatmap(pivot_gbs_rel, annot=True, fmt='.1f', cmap='RdBu_r', center=0,
                ax=axes[2,0], cbar_kws={'label': 'Rel. diff %'})
    axes[2,0].set_title('g_bs relative diff (800 vs 200 ppm)', fontsize=14, fontweight='bold')
    axes[2,0].set_xlabel('g_m', fontsize=12)
    axes[2,0].set_ylabel('f_Rdh', fontsize=12)

    sns.heatmap(pivot_vcmax_rel, annot=True, fmt='.1f', cmap='RdBu_r', center=0,
                ax=axes[2,1], cbar_kws={'label': 'Rel. diff %'})
    axes[2,1].set_title('V_cmax relative diff (800 vs 200 ppm)', fontsize=14, fontweight='bold')
    axes[2,1].set_xlabel('g_m', fontsize=12)
    axes[2,1].set_ylabel('f_Rdh', fontsize=12)

    sns.heatmap(pivot_abs_rel, annot=True, fmt='.1f', cmap='RdBu_r', center=0,
                ax=axes[2,2], cbar_kws={'label': 'Rel. diff %'})
    axes[2,2].set_title('A_bs relative diff (800 vs 200 ppm)', fontsize=14, fontweight='bold')
    axes[2,2].set_xlabel('g_m', fontsize=12)
    axes[2,2].set_ylabel('f_Rdh', fontsize=12)

    plt.tight_layout()
    heatmap_file = os.path.join(WORKING_DIR_FILEPATH, 'CO2_treatment_effect_heatmaps.png')
    plt.savefig(heatmap_file, dpi=150)
    plt.close()
    print(f"   Saved heatmaps: {heatmap_file}")

    # =============================================================================
    # 6. INTERACTION PLOTS FOR CO2 RESPONSES
    # =============================================================================

    print("📊 Creating interaction plots for CO2 responses...")

    fig, axes = plt.subplots(3, 3, figsize=(18, 14))

    # Plot absolute differences
    for i, (param, ylabel) in enumerate([
        ('g_bs_Value_diff', 'Δg_bs (800 - 200)'),
        ('V_cmax_Value_diff', 'ΔV_cmax (800 - 200)'),
        ('Abs_at_Ambient_Cm_diff', 'ΔA_bs (800 - 200)')
    ]):
        ax = axes[0, i]
        for f_rdh in sorted(df_diff['f_Rdh_used'].unique()):
            subset = df_diff[df_diff['f_Rdh_used'] == f_rdh]
            gm_means = subset.groupby('g_m_used')[param].mean()
            gm_std = subset.groupby('g_m_used')[param].std()
            ax.errorbar(gm_means.index, gm_means.values, yerr=gm_std.values,
                       marker='o', label=f'f_Rdh={f_rdh}', linewidth=2, markersize=8, capsize=5)
        ax.axhline(y=0, color='black', linestyle='--', alpha=0.3)
        ax.set_xlabel('g_m', fontsize=12)
        ax.set_ylabel(ylabel, fontsize=12)
        ax.set_title(f'{ylabel} by g_m and f_Rdh', fontsize=13, fontweight='bold')
        ax.legend()
        ax.grid(True, alpha=0.3)

    # Plot percent changes
    for i, (param, ylabel) in enumerate([
        ('g_bs_Value_pct_change', 'g_bs % change'),
        ('V_cmax_Value_pct_change', 'V_cmax % change'),
        ('Abs_at_Ambient_Cm_pct_change', 'A_bs % change')
    ]):
        ax = axes[1, i]
        for f_rdh in sorted(df_diff['f_Rdh_used'].unique()):
            subset = df_diff[df_diff['f_Rdh_used'] == f_rdh]
            gm_means = subset.groupby('g_m_used')[param].mean()
            gm_std = subset.groupby('g_m_used')[param].std()
            ax.errorbar(gm_means.index, gm_means.values, yerr=gm_std.values,
                       marker='o', label=f'f_Rdh={f_rdh}', linewidth=2, markersize=8, capsize=5)
        ax.axhline(y=0, color='black', linestyle='--', alpha=0.3)
        ax.set_xlabel('g_m', fontsize=12)
        ax.set_ylabel(ylabel, fontsize=12)
        ax.set_title(f'{ylabel} by g_m and f_Rdh', fontsize=13, fontweight='bold')
        ax.legend()
        ax.grid(True, alpha=0.3)

    # Plot relative differences
    for i, (param, ylabel) in enumerate([
        ('g_bs_Value_rel_diff', 'g_bs rel. diff %'),
        ('V_cmax_Value_rel_diff', 'V_cmax rel. diff %'),
        ('Abs_at_Ambient_Cm_rel_diff', 'A_bs rel. diff %')
    ]):
        ax = axes[2, i]
        for f_rdh in sorted(df_diff['f_Rdh_used'].unique()):
            subset = df_diff[df_diff['f_Rdh_used'] == f_rdh]
            gm_means = subset.groupby('g_m_used')[param].mean()
            gm_std = subset.groupby('g_m_used')[param].std()
            ax.errorbar(gm_means.index, gm_means.values, yerr=gm_std.values,
                       marker='o', label=f'f_Rdh={f_rdh}', linewidth=2, markersize=8, capsize=5)
        ax.axhline(y=0, color='black', linestyle='--', alpha=0.3)
        ax.set_xlabel('g_m', fontsize=12)
        ax.set_ylabel(ylabel, fontsize=12)
        ax.set_title(f'{ylabel} by g_m and f_Rdh', fontsize=13, fontweight='bold')
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    interaction_file = os.path.join(WORKING_DIR_FILEPATH, 'CO2_treatment_interaction_plots.png')
    plt.savefig(interaction_file, dpi=150)
    plt.close()
    print(f"   Saved interaction plots: {interaction_file}")

    # =============================================================================
    # 7. SPECIES-LEVEL COMPARISON
    # =============================================================================

    print("\n📊 Creating species-level CO2 response comparison...")

    # Calculate mean response per species across all parameter combinations
    species_summary = df_diff.groupby('Species').agg({
        'g_bs_Value_diff': ['mean', 'std'],
        'V_cmax_Value_diff': ['mean', 'std'],
        'Abs_at_Ambient_Cm_diff': ['mean', 'std']
    }).round(4)

    species_file = os.path.join(WORKING_DIR_FILEPATH, 'CO2_response_by_species.csv')
    species_summary.to_csv(species_file)
    print(f"   Saved species summary: {species_file}")

    # Plot species comparison
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 6))

    species_means = df_diff.groupby('Species')[['g_bs_Value_diff', 'V_cmax_Value_diff', 'Abs_at_Ambient_Cm_diff']].mean()
    species_means = species_means.sort_values('g_bs_Value_diff')

    species_means['g_bs_Value_diff'].plot(kind='barh', ax=ax1, color='steelblue')
    ax1.axvline(x=0, color='black', linestyle='--', alpha=0.3)
    ax1.set_xlabel('Mean Δg_bs (800 - 200 ppm)', fontsize=12)
    ax1.set_title('CO2 Effect on g_bs by Species', fontsize=14, fontweight='bold')
    ax1.grid(True, alpha=0.3, axis='x')

    species_means = species_means.sort_values('V_cmax_Value_diff')
    species_means['V_cmax_Value_diff'].plot(kind='barh', ax=ax2, color='coral')
    ax2.axvline(x=0, color='black', linestyle='--', alpha=0.3)
    ax2.set_xlabel('Mean ΔV_cmax (800 - 200 ppm)', fontsize=12)
    ax2.set_title('CO2 Effect on V_cmax by Species', fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3, axis='x')

    species_means = species_means.sort_values('Abs_at_Ambient_Cm_diff')
    species_means['Abs_at_Ambient_Cm_diff'].plot(kind='barh', ax=ax3, color='forestgreen')
    ax3.axvline(x=0, color='black', linestyle='--', alpha=0.3)
    ax3.set_xlabel('Mean ΔA_bs (800 - 200 ppm)', fontsize=12)
    ax3.set_title('CO2 Effect on A_bs by Species', fontsize=14, fontweight='bold')
    ax3.grid(True, alpha=0.3, axis='x')

    plt.tight_layout()
    species_plot = os.path.join(WORKING_DIR_FILEPATH, 'CO2_response_by_species.png')
    plt.savefig(species_plot, dpi=150)
    plt.close()
    print(f"   Saved species comparison: {species_plot}")

    # =============================================================================
    # 8. FOREST PLOTS: 3-PANEL (3 ROWS FOR f_Rdh) WITH g_m OVERLAID
    # =============================================================================

    print("\n📊 Creating forest plots (3 panels per parameter, g_m overlaid)...")

    # Color scheme for g_m values (already have gm_values, f_rdh_values, species_list defined)
    colors = {gm_values[0]: '#2E86AB', gm_values[1]: '#A23B72', gm_values[2]: '#F18F01'}  # Blue, Purple, Orange
    markers = {gm_values[0]: 'o', gm_values[1]: 's', gm_values[2]: '^'}  # Circle, Square, Triangle

    # Dodge amount for overlaying points
    dodge = 0.2

    # Create forest plot for each difference metric and parameter
    for param_base in ['g_bs_Value', 'V_cmax_Value', 'Abs_at_Ambient_Cm']:
        for metric_suffix, metric_name in [('_diff', 'Absolute Difference'),
                                            ('_pct_change', 'Percent Change'),
                                            ('_rel_diff', 'Relative Difference')]:
            param_col = param_base + metric_suffix

            if param_col not in df_diff.columns:
                continue

            # Create figure with 3 rows (one per f_Rdh), single column
            fig, axes = plt.subplots(3, 1, figsize=(12, 16))
            fig.suptitle(f'{param_base.replace("_Value", "")} - {metric_name} (800 vs 200 ppm)',
                        fontsize=16, fontweight='bold', y=0.996)

            for i, f_rdh in enumerate(f_rdh_values):
                ax = axes[i]

                # Collect all species present at this f_Rdh level
                all_species_this_panel = set()
                for gm in gm_values:
                    subset = df_diff[(df_diff['g_m_used'] == gm) & (df_diff['f_Rdh_used'] == f_rdh)]
                    all_species_this_panel.update(subset['Species'].values)

                # Sort species alphabetically for consistent ordering
                all_species_sorted = sorted(list(all_species_this_panel))

                if len(all_species_sorted) > 0:
                    # Create y positions for species
                    y_positions = {sp: idx for idx, sp in enumerate(all_species_sorted)}

                    # Track which g_m values we've seen for legend
                    legend_added = set()

                    # Plot each g_m value with different color and dodge
                    for gm_idx, gm in enumerate(gm_values):
                        subset = df_diff[(df_diff['g_m_used'] == gm) & (df_diff['f_Rdh_used'] == f_rdh)]
                        subset = subset.sort_values('Species')

                        if len(subset) > 0:
                            # Calculate dodge offset for this g_m
                            offset = (gm_idx - 1) * dodge  # -dodge, 0, +dodge for 3 values

                            for _, row in subset.iterrows():
                                species = row['Species']
                                value = row[param_col]
                                y_pos = y_positions[species] + offset

                                # Add label only once per g_m for legend
                                add_label = gm not in legend_added
                                if add_label:
                                    legend_added.add(gm)

                                # Plot point
                                ax.scatter(value, y_pos, s=150,
                                          color=colors[gm],
                                          marker=markers[gm],
                                          edgecolors='black',
                                          linewidth=1.5,
                                          zorder=3,
                                          alpha=0.85,
                                          label=f'g_m = {gm}' if add_label else '')

                                # Add connecting line to zero
                                ax.plot([0, value], [y_pos, y_pos],
                                       color=colors[gm],
                                       linewidth=1.5,
                                       alpha=0.25,
                                       zorder=2)

                    # Add vertical line at zero
                    ax.axvline(x=0, color='red', linestyle='--', linewidth=2.5, alpha=0.6, zorder=1)

                    # Formatting
                    ax.set_yticks(range(len(all_species_sorted)))
                    ax.set_yticklabels(all_species_sorted, fontsize=11, style='italic')
                    ax.set_xlabel(metric_name, fontsize=12, fontweight='bold')
                    ax.set_title(f'f_Rdh = {f_rdh}', fontsize=14, fontweight='bold', pad=10)
                    ax.grid(True, alpha=0.2, axis='x', linewidth=0.8)
                    ax.set_ylim(-0.6, len(all_species_sorted) - 0.4)

                    # Invert y-axis so first species is at top
                    ax.invert_yaxis()

                    # Add light background shading
                    ax.set_facecolor('#f8f9fa')

                    # Add legend (only for first panel)
                    if i == 0:
                        ax.legend(loc='best', frameon=True, fancybox=True,
                                 shadow=True, fontsize=10, ncol=3)
                else:
                    ax.text(0.5, 0.5, 'No data', ha='center', va='center',
                           transform=ax.transAxes, fontsize=14, style='italic')
                    ax.set_title(f'f_Rdh = {f_rdh}', fontsize=14, fontweight='bold')

            plt.tight_layout()
            forest_file = os.path.join(WORKING_DIR_FILEPATH,
                                      f'forest_plot_{param_base}_{metric_suffix[1:]}.png')
            plt.savefig(forest_file, dpi=150, bbox_inches='tight')
            plt.close()
            print(f"   Saved: forest_plot_{param_base}_{metric_suffix[1:]}.png")

    # =============================================================================
    # 9. FINAL SUMMARY REPORT
    # =============================================================================

    print("\n" + "="*80)
    print("📋 FINAL SUMMARY: CO2 TREATMENT EFFECT ANALYSIS")
    print("="*80)

    print(f"\n✅ Analysis complete!")
    print(f"\nFiles created:")
    print(f"   1. CO2_treatment_differences.csv - Per-species CO2 differences (plants pooled)")
    print(f"   2. CO2_treatment_summary_by_params.csv - Summary by g_m and f_Rdh")
    print(f"   3. CO2_response_ANOVA.csv - Statistical tests (ANOVA)")
    print(f"   4. CO2_response_by_species.csv - Species-level summary")
    print(f"   5. CO2_treatment_effect_heatmaps.png - Heatmaps (abs, %, rel. diff)")
    print(f"   6. CO2_treatment_interaction_plots.png - Interaction plots (abs, %, rel. diff)")
    print(f"   7. CO2_response_by_species.png - Species comparison bar charts")
    print(f"   8. forest_plot_g_bs_Value_*.png - Forest plots for g_bs (3 panels, g_m overlaid)")
    print(f"   9. forest_plot_V_cmax_Value_*.png - Forest plots for V_cmax (3 panels, g_m overlaid)")
    print(f"  10. forest_plot_Abs_at_Ambient_Cm_*.png - Forest plots for A_bs (3 panels, g_m overlaid)")
    print(f"\n      (3 forest plots per parameter: diff, pct_change, rel_diff = 9 total)")
    print(f"      (Each forest plot: 3 rows for f_Rdh, colors/shapes for g_m)")

    print(f"\n📊 Difference metrics calculated:")
    print(f"   • Absolute difference: 800 - 200 ppm")
    print(f"   • Percent change: (800 - 200) / 200 × 100")
    print(f"   • Relative difference: (800 - 200) / mean(200, 800) × 100 (symmetric)")

    print(f"\n📋 Analysis structure:")
    print(f"   • Per-species differences (plants pooled within species)")
    print(f"   • {len(species_list)} species analyzed")
    print(f"   • {len(gm_values)} g_m values × {len(f_rdh_values)} f_Rdh values = {len(gm_values) * len(f_rdh_values)} combinations")
    print(f"   • Forest plots: 3 panels (f_Rdh rows), g_m values overlaid with colors/shapes")
    print(f"   • Statistical tests: One-way ANOVA for g_m and f_Rdh effects")

    print(f"\n🔍 Key findings:")

    # Overall mean responses
    overall_means = df_diff[['g_bs_Value_diff', 'V_cmax_Value_diff', 'Abs_at_Ambient_Cm_diff']].mean()
    print(f"\nOverall mean CO2 responses (800 - 200 ppm):")
    print(f"   Δg_bs: {overall_means['g_bs_Value_diff']:.4f}")
    print(f"   ΔV_cmax: {overall_means['V_cmax_Value_diff']:.2f}")
    print(f"   ΔA_bs: {overall_means['Abs_at_Ambient_Cm_diff']:.2f}")

    # Variability across parameter assumptions
    gm_variability = df_diff.groupby('g_m_used')['g_bs_Value_diff'].mean()
    frdh_variability = df_diff.groupby('f_Rdh_used')['g_bs_Value_diff'].mean()

    print(f"\nMean Δg_bs by g_m assumption:")
    for gm, val in gm_variability.items():
        print(f"   g_m = {gm}: {val:.4f}")

    print(f"\nMean Δg_bs by f_Rdh assumption:")
    for frdh, val in frdh_variability.items():
        print(f"   f_Rdh = {frdh}: {val:.4f}")

    if stats_results:
        print(f"\n⚠️ Significant effects found:")
        sig_results = [r for r in stats_results if r['Significant'] == 'Yes']
        if sig_results:
            for r in sig_results:
                print(f"   {r['Parameter']} is affected by {r['Factor']} (p = {r['p_value']:.4f})")
        else:
            print(f"   None - CO2 responses are robust to parameter assumptions!")

    print("\n" + "="*80)

📊 ANALYZING CO2 TREATMENT EFFECTS ACROSS PARAMETER COMBINATIONS

✅ Loaded 810 total fits
   Successful fits: 810 (100.0%)
   g_m values: [np.float64(0.3), np.float64(0.6), np.float64(0.9)]
   f_Rdh values: [np.float64(0.0), np.float64(0.1), np.float64(0.2)]
   Species: 11

📈 Calculating CO2 treatment differences per species (pooling plants)...
   Calculated differences for 99 species × parameter combinations
   Species: 11
   Parameter combinations (g_m × f_Rdh): 9

✅ Saved CO2 treatment differences to: gdrive/My Drive/Colab Notebooks/CO2_treatment_differences.csv

📊 Summary statistics by g_m and f_Rdh combination:
--------------------------------------------------------------------------------
                    g_bs_Value_diff               V_cmax_Value_diff           \
                               mean     std count              mean      std   
g_m_used f_Rdh_used                                                            
0.3      0.0                 0.0042  0.0122    11       